# SeSE: Semantic Structural Entropy — Long-Form Uncertainty Quantification

Implementation of **SeSE** (Semantic Structural Entropy) for **claim-level uncertainty estimation** in long-form LLM generation, based on:

> *SeSE: Black-Box Uncertainty Quantification for Large Language Models Based on Structural Information Theory* (Zhao et al., 2026)

## Pipeline Overview
1. **Response Sampling** — Sample N stochastic responses from a local HuggingFace LLM
2. **Claim Decomposition** — Break the greedy response into atomic claims
3. **Bipartite Graph Construction** — Build a claim-response entailment graph
4. **Hierarchical Abstraction** — Find the optimal K-dimensional encoding tree via structural entropy minimization
5. **Claim-level SeSE** — Compute per-claim uncertainty from root-to-leaf path entropy

## 0. Install Dependencies

In [ ]:
!pip install transformers torch accelerate sentencepiece networkx matplotlib seaborn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 24.1 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 21.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 39.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 26.3 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 32.5 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 45.1 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 48.7 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 39.7 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 48.1 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 46.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 29.3 MB/

## 1. Imports & Configuration

In [1]:
import torch
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import json
import re
import warnings
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    pipeline,
)

warnings.filterwarnings('ignore')

# ─── Configuration ────────────────────────────────────────────────────────────
@dataclass
class SeSEConfig:
    # Generation model (change to any HuggingFace causal LM)
    gen_model_name: str = "mistralai/Mistral-7B-Instruct-v0.2"
    # NLI model for entailment judgment
    nli_model_name: str = "cross-encoder/nli-deberta-v3-large"
    # Number of stochastic samples
    n_samples: int = 10
    # Sampling temperature
    temperature: float = 1.0
    # Top-p nucleus sampling
    top_p: float = 0.95
    # Top-k sampling
    top_k: int = 20
    # Max new tokens for generation
    max_new_tokens: int = 256
    # Max encoding tree height
    K: int = 3
    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    # Entailment threshold (for binary bipartite graph edge)
    entailment_threshold: float = 0.5

cfg = SeSEConfig()
print(f"Device: {cfg.device}")
print(f"Generation model: {cfg.gen_model_name}")
print(f"NLI model: {cfg.nli_model_name}")

ModuleNotFoundError: No module named 'torch'

## 2. Load Models

In [ ]:
# ─── Generation Model ─────────────────────────────────────────────────────────
print("Loading generation model...")
gen_tokenizer = AutoTokenizer.from_pretrained(cfg.gen_model_name)
gen_tokenizer.pad_token = gen_tokenizer.eos_token

gen_model = AutoModelForCausalLM.from_pretrained(
    cfg.gen_model_name,
    torch_dtype=torch.float16 if cfg.device == "cuda" else torch.float32,
    device_map="auto" if cfg.device == "cuda" else None,
)
gen_model.eval()

# ─── NLI Model ────────────────────────────────────────────────────────────────
print("Loading NLI model...")
nli_pipe = pipeline(
    "text-classification",
    model=cfg.nli_model_name,
    device=0 if cfg.device == "cuda" else -1,
    top_k=None,   # return all labels
)
print("Models loaded.")

## 3. Step 1 — Response Sampling

In [ ]:
def format_prompt(question: str) -> str:
    """Format a question into an instruction-following prompt."""
    return f"""[INST] You are a helpful assistant. Please write a detailed paragraph about the following topic.

Topic: {question}

Provide a comprehensive paragraph with specific facts. [/INST]"""


def sample_responses(
    question: str,
    n_samples: int = cfg.n_samples,
    temperature: float = cfg.temperature,
    top_p: float = cfg.top_p,
    top_k: int = cfg.top_k,
    max_new_tokens: int = cfg.max_new_tokens,
) -> Tuple[str, List[str]]:
    """
    Generate one greedy (T=0) response and N stochastic responses.

    Returns:
        greedy_response (str): The most confident answer.
        sampled_responses (List[str]): N stochastic answers for uncertainty modeling.
    """
    prompt = format_prompt(question)
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(cfg.device)
    input_len = inputs["input_ids"].shape[1]

    def decode(output_ids):
        tokens = output_ids[input_len:]
        return gen_tokenizer.decode(tokens, skip_special_tokens=True).strip()

    # Greedy decode (T=0)
    with torch.no_grad():
        greedy_out = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    greedy_response = decode(greedy_out[0])

    # Stochastic samples
    sampled_responses = []
    for _ in range(n_samples):
        with torch.no_grad():
            out = gen_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                pad_token_id=gen_tokenizer.eos_token_id,
            )
        sampled_responses.append(decode(out[0]))

    return greedy_response, sampled_responses


# ─── Quick test ──────────────────────────────────────────────────────────────
question = "Tell me about Marie Curie"
print(f"Question: {question}\n")
greedy_response, sampled_responses = sample_responses(question)

print("=== Greedy Response ===")
print(greedy_response[:500], "...")
print(f"\n=== {len(sampled_responses)} Stochastic Samples (first shown) ===")
print(sampled_responses[0][:300], "...")

## 4. Step 2 — Claim Decomposition
Break the greedy response into **atomic claims** — the smallest semantically independent units.

In [ ]:
CLAIM_DECOMP_PROMPT = """[INST] You will be provided with a long-form text that contains multiple claims.
A claim is the smallest independent and self-contained perspective.
Your task is to precisely identify and extract each claim within the given text, making sure there is no semantic repetition.
Then, for the sake of clarity, resolve all anaphora (pronouns or other referring expressions) within the claims.
Each claim should be concise and independently complete.
Ensure that you are comprehensive and list each claim as a separate sentence.

Return ONLY a JSON array of strings, with no other text. Example format:
["Claim one.", "Claim two.", "Claim three."]

The input is: {text}
[/INST]"""


def decompose_claims(text: str) -> List[str]:
    """
    Decompose a long-form response into atomic claims using the LLM itself.
    Falls back to sentence splitting if JSON parsing fails.
    """
    prompt = CLAIM_DECOMP_PROMPT.format(text=text)
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(cfg.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = gen_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=gen_tokenizer.eos_token_id,
        )
    raw = gen_tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()

    # Parse JSON output
    try:
        # Find JSON array in output
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        if match:
            claims = json.loads(match.group())
            claims = [c.strip() for c in claims if isinstance(c, str) and c.strip()]
            if claims:
                return claims
    except (json.JSONDecodeError, AttributeError):
        pass

    # Fallback: split on sentence boundaries
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 10]


# ─── Decompose the greedy response ───────────────────────────────────────────
claims = decompose_claims(greedy_response)
print(f"Extracted {len(claims)} atomic claims from greedy response:\n")
for i, claim in enumerate(claims, 1):
    print(f"  [{i}] {claim}")

## 5. Step 3 — Bipartite Graph Construction
Build a claim-response bipartite graph $G_{cr} = ((R, C), E)$.
An edge $(r, c) \in E$ exists (weight=1) if response $r$ **entails** claim $c$.

In [ ]:
ENTAILMENT_LABEL_MAP = {
    "entailment": 1,
    "ENTAILMENT": 1,
    "neutral": 0,
    "NEUTRAL": 0,
    "contradiction": -1,
    "CONTRADICTION": -1,
}


def check_entailment_nli(
    response: str,
    claim: str,
    threshold: float = cfg.entailment_threshold,
) -> bool:
    """
    Check if `response` entails `claim` using the NLI model.
    Returns True if entailment probability > threshold.
    """
    # NLI takes (premise, hypothesis)
    result = nli_pipe(
        {"text": response[:512], "text_pair": claim},
        truncation=True,
    )
    # result is a list of {label: ..., score: ...}
    scores = {item["label"].lower(): item["score"] for item in result}
    entail_score = scores.get("entailment", 0.0)
    return entail_score > threshold


def build_bipartite_graph(
    responses: List[str],
    claims: List[str],
) -> Tuple[nx.Graph, np.ndarray]:
    """
    Build claim-response bipartite graph.

    Nodes:
        r_0, r_1, ... r_{N-1}  — response nodes
        c_0, c_1, ... c_{M-1}  — claim nodes

    Edges:
        (r_i, c_j) with weight=1 if response_i entails claim_j

    Returns:
        G: networkx bipartite graph
        adj: (N x M) binary adjacency matrix
    """
    N = len(responses)
    M = len(claims)
    adj = np.zeros((N, M), dtype=np.float32)

    G = nx.Graph()
    r_nodes = [f"r_{i}" for i in range(N)]
    c_nodes = [f"c_{j}" for j in range(M)]

    for r_node in r_nodes:
        G.add_node(r_node, bipartite=0, type="response")
    for c_node in c_nodes:
        G.add_node(c_node, bipartite=1, type="claim")

    print(f"Building bipartite graph ({N} responses × {M} claims)...")
    for i, (r_node, response) in enumerate(zip(r_nodes, responses)):
        for j, (c_node, claim) in enumerate(zip(c_nodes, claims)):
            entails = check_entailment_nli(response, claim)
            if entails:
                G.add_edge(r_node, c_node, weight=1)
                adj[i, j] = 1.0
        print(f"  Response {i+1}/{N} processed", end="\r")

    total_edges = G.number_of_edges()
    print(f"\nGraph built: {G.number_of_nodes()} nodes, {total_edges} edges")
    return G, adj, r_nodes, c_nodes


# ─── Build the graph ─────────────────────────────────────────────────────────
G_cr, adj_matrix, r_nodes, c_nodes = build_bipartite_graph(sampled_responses, claims)

# Visualize adjacency matrix
fig, ax = plt.subplots(figsize=(max(6, len(claims)), 4))
sns.heatmap(
    adj_matrix,
    ax=ax,
    cmap="Blues",
    cbar=False,
    linewidths=0.5,
    xticklabels=[f"C{j}" for j in range(len(claims))],
    yticklabels=[f"R{i}" for i in range(len(sampled_responses))],
    vmin=0, vmax=1,
)
ax.set_title("Bipartite Graph Adjacency Matrix\n(rows=responses, cols=claims, blue=entailment)",
             fontsize=12, fontweight='bold')
ax.set_xlabel("Claims"); ax.set_ylabel("Responses")
plt.tight_layout()
plt.show()

print(f"\nClaim support counts (number of responses that entail each claim):")
for j, claim in enumerate(claims):
    count = int(adj_matrix[:, j].sum())
    bar = '█' * count + '░' * (len(sampled_responses) - count)
    print(f"  C{j} [{bar}] {count}/{len(sampled_responses)}  {claim[:60]}...")

## 6. Step 4 — Hierarchical Abstraction via Structural Entropy Minimization

We build an optimal K-dimensional encoding tree $\mathcal{T}^*$ by greedily applying **merge** and **combine** operators that minimize structural entropy.

**Structural entropy of node $\alpha$:**
$$H^\mathcal{T}(G;\alpha) = -\frac{g_\alpha}{\text{vol}(G)} \log_2 \frac{V_\alpha}{V_{\alpha^-}}$$

In [ ]:
# ─── Encoding Tree Node ───────────────────────────────────────────────────────
@dataclass
class TreeNode:
    """A node in the encoding tree."""
    node_id: str
    members: List[str]          # graph node IDs in this partition
    parent: Optional[str] = None
    children: List[str] = field(default_factory=list)
    depth: int = 0


class EncodingTree:
    """
    Encoding tree for structural entropy computation.
    Supports merge and combine operators from Li & Pan (2016).
    """

    def __init__(self, G: nx.Graph):
        """
        Initialize 1-dimensional encoding tree:
          root → one leaf per graph node.
        """
        self.G = G
        self.all_nodes_list = list(G.nodes())
        self.vol = self._compute_volume(self.all_nodes_list)

        # Compute stationary distribution via degree (undirected case)
        self.pi = self._stationary_distribution()

        # Initialize tree
        self.nodes: Dict[str, TreeNode] = {}
        self._counter = 0

        root_id = "root"
        self.root_id = root_id
        self.nodes[root_id] = TreeNode(
            node_id=root_id,
            members=self.all_nodes_list[:],
            parent=None,
            depth=0,
        )

        # One leaf per graph node directly under root
        for gn in self.all_nodes_list:
            leaf_id = f"leaf_{gn}"
            self.nodes[leaf_id] = TreeNode(
                node_id=leaf_id,
                members=[gn],
                parent=root_id,
                depth=1,
            )
            self.nodes[root_id].children.append(leaf_id)

    # ── Volume & Stationary Distribution ─────────────────────────────────────
    def _compute_volume(self, members: List[str]) -> float:
        """Sum of weighted degrees for all graph nodes."""
        return sum(
            sum(d.get("weight", 1) for d in self.G[n].values())
            for n in self.G.nodes()
        ) or 1.0

    def _stationary_distribution(self) -> Dict[str, float]:
        """π(v) = degree(v) / vol(G) for undirected weighted graph."""
        degrees = {}
        for v in self.G.nodes():
            degrees[v] = sum(d.get("weight", 1) for d in self.G[v].values())
        vol = sum(degrees.values()) or 1.0
        return {v: degrees[v] / vol for v in degrees}

    def _V_alpha(self, members: List[str]) -> float:
        """Volume of the sub-graph induced by members (sum of π over transitions into members)."""
        member_set = set(members)
        val = 0.0
        for v in self.G.nodes():
            for u, data in self.G[v].items():
                if u in member_set:
                    val += self.pi.get(v, 0) * data.get("weight", 1)
        return val or 1e-12

    def _g_alpha(self, members: List[str]) -> float:
        """Edges from outside members into members (cut weight)."""
        member_set = set(members)
        val = 0.0
        for v in self.G.nodes():
            if v not in member_set:
                for u, data in self.G[v].items():
                    if u in member_set:
                        val += self.pi.get(v, 0) * data.get("weight", 1)
        return val

    def _H_alpha(self, node_id: str) -> float:
        """Structural entropy contribution of a single non-root node."""
        node = self.nodes[node_id]
        parent = self.nodes[node.parent]
        g = self._g_alpha(node.members)
        V_alpha = self._V_alpha(node.members)
        V_parent = self._V_alpha(parent.members)
        if V_parent < 1e-12 or V_alpha < 1e-12:
            return 0.0
        return -(g / self.vol) * np.log2(V_alpha / V_parent + 1e-12)

    def total_entropy(self) -> float:
        """Total structural entropy H^T(G) = sum over all non-root nodes."""
        return sum(
            self._H_alpha(nid)
            for nid, node in self.nodes.items()
            if node.parent is not None  # skip root
        )

    # ── Sibling Enumeration ───────────────────────────────────────────────────
    def _get_sibling_pairs(self) -> List[Tuple[str, str]]:
        """Return all pairs of nodes that share the same parent."""
        pairs = []
        parent_children: Dict[str, List[str]] = {}
        for nid, node in self.nodes.items():
            if node.parent:
                parent_children.setdefault(node.parent, []).append(nid)
        for children in parent_children.values():
            for i in range(len(children)):
                for j in range(i + 1, len(children)):
                    pairs.append((children[i], children[j]))
        return pairs

    def _current_height(self) -> int:
        return max((node.depth for node in self.nodes.values()), default=0)

    # ── Merge Operator ────────────────────────────────────────────────────────
    def _delta_merge(self, alpha_id: str, beta_id: str) -> float:
        """
        Entropy change from merging leaf siblings α and β into a new parent μ.
        Δ = H(α) + H(β) - H(μ_new) - ΔH due to edge re-routing.
        Simplified: reduction = cross-cut savings.
        """
        alpha = self.nodes[alpha_id]
        beta = self.nodes[beta_id]
        combined = alpha.members + beta.members

        g_alpha = self._g_alpha(alpha.members)
        g_beta = self._g_alpha(beta.members)
        V_alpha = self._V_alpha(alpha.members)
        V_beta = self._V_alpha(beta.members)
        V_merged = self._V_alpha(combined)
        V_parent = self._V_alpha(self.nodes[alpha.parent].members)

        # Cross-cut between alpha and beta
        g_ab = sum(
            self.pi.get(v, 0) * data.get("weight", 1)
            for v in alpha.members
            for u, data in self.G[v].items()
            if u in set(beta.members)
        )
        g_ba = sum(
            self.pi.get(v, 0) * data.get("weight", 1)
            for v in beta.members
            for u, data in self.G[v].items()
            if u in set(alpha.members)
        )

        g_merged = self._g_alpha(combined)

        h_before = self._H_alpha(alpha_id) + self._H_alpha(beta_id)
        h_merged = -(g_merged / self.vol) * np.log2(V_merged / (V_parent + 1e-12) + 1e-12)
        return h_before - h_merged

    def _apply_merge(self, alpha_id: str, beta_id: str) -> str:
        """Merge two leaf siblings into a new intermediate node."""
        self._counter += 1
        mu_id = f"merge_{self._counter}"
        alpha = self.nodes[alpha_id]
        beta = self.nodes[beta_id]
        parent_id = alpha.parent
        parent = self.nodes[parent_id]

        new_node = TreeNode(
            node_id=mu_id,
            members=alpha.members + beta.members,
            parent=parent_id,
            children=[alpha_id, beta_id],
            depth=alpha.depth,
        )
        self.nodes[mu_id] = new_node

        # Update depths of merged nodes
        alpha.parent = mu_id
        beta.parent = mu_id
        alpha.depth += 1
        beta.depth += 1

        # Update parent's children
        parent.children = [
            c for c in parent.children if c not in (alpha_id, beta_id)
        ] + [mu_id]
        return mu_id

    # ── Combine Operator ──────────────────────────────────────────────────────
    def _delta_combine(self, alpha_id: str, beta_id: str) -> float:
        """Entropy change from combining two subtree siblings under a new parent μ."""
        alpha = self.nodes[alpha_id]
        beta = self.nodes[beta_id]
        combined = alpha.members + beta.members

        g_alpha = self._g_alpha(alpha.members)
        g_beta = self._g_alpha(beta.members)
        g_combined = self._g_alpha(combined)
        V_parent = self._V_alpha(self.nodes[alpha.parent].members)
        V_combined = self._V_alpha(combined)

        h_before = self._H_alpha(alpha_id) + self._H_alpha(beta_id)
        # After combine: mu gets H_mu, alpha/beta keep their sub-entropy unchanged
        h_mu = -(g_combined / self.vol) * np.log2(V_combined / (V_parent + 1e-12) + 1e-12)
        # α and β now measure against μ
        h_alpha_new = -(g_alpha / self.vol) * np.log2(self._V_alpha(alpha.members) / (V_combined + 1e-12) + 1e-12)
        h_beta_new = -(g_beta / self.vol) * np.log2(self._V_alpha(beta.members) / (V_combined + 1e-12) + 1e-12)
        h_after = h_mu + h_alpha_new + h_beta_new
        return h_before - h_after

    def _apply_combine(self, alpha_id: str, beta_id: str) -> str:
        """Combine two siblings under a new intermediate node."""
        self._counter += 1
        mu_id = f"combine_{self._counter}"
        alpha = self.nodes[alpha_id]
        beta = self.nodes[beta_id]
        parent_id = alpha.parent
        parent = self.nodes[parent_id]

        new_node = TreeNode(
            node_id=mu_id,
            members=alpha.members + beta.members,
            parent=parent_id,
            children=[alpha_id, beta_id],
            depth=alpha.depth,
        )
        self.nodes[mu_id] = new_node
        alpha.parent = mu_id
        beta.parent = mu_id
        alpha.depth += 1
        beta.depth += 1

        parent.children = [
            c for c in parent.children if c not in (alpha_id, beta_id)
        ] + [mu_id]
        return mu_id

    # ── Main Optimization (Algorithm 1) ───────────────────────────────────────
    def optimize(self, K: int = cfg.K) -> float:
        """
        Greedily apply merge/combine operators to minimize structural entropy,
        up to tree height K.

        Returns final structural entropy.
        """
        print(f"Optimizing encoding tree (K={K})...")
        for iteration in range(1000):  # safety cap
            if self._current_height() >= K:
                break

            pairs = self._get_sibling_pairs()
            if not pairs:
                break

            # Try merge
            best_merge = max(
                ((self._delta_merge(a, b), a, b) for a, b in pairs),
                key=lambda x: x[0], default=(0, None, None)
            )
            # Try combine
            best_combine = max(
                ((self._delta_combine(a, b), a, b) for a, b in pairs),
                key=lambda x: x[0], default=(0, None, None)
            )

            dm, am, bm = best_merge
            dc, ac, bc = best_combine

            if dm <= 0 and dc <= 0:
                break  # no improvement possible

            if dm >= dc and dm > 0 and am is not None:
                self._apply_merge(am, bm)
            elif dc > 0 and ac is not None:
                self._apply_combine(ac, bc)
            else:
                break

        final_entropy = self.total_entropy()
        print(f"Optimization complete. Final structural entropy: {final_entropy:.4f}")
        return final_entropy

    # ── Leaf Lookup ───────────────────────────────────────────────────────────
    def find_leaf(self, graph_node_id: str) -> Optional[str]:
        """Return the tree node ID of the leaf containing the given graph node."""
        for nid, node in self.nodes.items():
            if node.members == [graph_node_id]:
                return nid
        return None

    def path_to_root(self, leaf_id: str) -> List[str]:
        """Return path from root down to leaf (inclusive, root first)."""
        path = []
        nid = leaf_id
        while nid is not None:
            path.append(nid)
            nid = self.nodes[nid].parent
        return list(reversed(path))


print("EncodingTree class defined successfully.")

## 7. Step 5 — Claim-Level SeSE Computation
$$\text{SeSE}(G_{cr}; c) = -\sum_{\alpha \in P(\lambda \to \gamma) \setminus \{\lambda\}} \frac{g_\alpha}{\text{vol}(G)} \log_2 \frac{V_\alpha}{V_{\alpha^-}}$$

In [ ]:
def compute_sese_longform(
    G_cr: nx.Graph,
    c_nodes: List[str],
    claims: List[str],
    K: int = cfg.K,
) -> Dict[str, float]:
    """
    Compute claim-level SeSE scores for all claims in the bipartite graph.

    Args:
        G_cr: bipartite claim-response graph
        c_nodes: list of claim node IDs (e.g. ['c_0', 'c_1', ...])
        claims: list of claim strings
        K: max encoding tree height

    Returns:
        dict mapping claim string → SeSE score
    """
    # Build and optimize encoding tree
    tree = EncodingTree(G_cr)
    tree.optimize(K=K)

    scores = {}
    for c_node, claim in zip(c_nodes, claims):
        leaf_id = tree.find_leaf(c_node)
        if leaf_id is None:
            scores[claim] = float('nan')
            continue

        # Path from root to leaf, excluding root
        path = tree.path_to_root(leaf_id)[1:]  # drop root

        # Sum structural entropy along path
        sese_score = sum(tree._H_alpha(nid) for nid in path)
        scores[claim] = sese_score

    return scores, tree


# ─── Run SeSE ─────────────────────────────────────────────────────────────────
sese_scores, encoding_tree = compute_sese_longform(G_cr, c_nodes, claims, K=cfg.K)

print("\n" + "="*70)
print("CLAIM-LEVEL SeSE SCORES")
print("="*70)
print(f"{'Score':>8}  {'Support':>8}  Claim")
print("-"*70)

sorted_claims = sorted(sese_scores.items(), key=lambda x: x[1])
for claim, score in sorted_claims:
    c_idx = claims.index(claim)
    support = int(adj_matrix[:, c_idx].sum())
    label = "✓ CERTAIN" if score < np.nanmedian(list(sese_scores.values())) else "? UNCERTAIN"
    print(f"{score:8.4f}  {support:>5}/{len(sampled_responses)}  [{label}] {claim[:55]}")

## 8. Visualization

In [ ]:
def plot_sese_results(claims, sese_scores, adj_matrix, sampled_responses):
    """Comprehensive visualization of SeSE results."""
    M = len(claims)
    scores = [sese_scores.get(c, float('nan')) for c in claims]
    support = [int(adj_matrix[:, j].sum()) for j in range(M)]
    support_ratio = [s / len(sampled_responses) for s in support]
    median_score = np.nanmedian(scores)
    colors = ['#2ecc71' if s < median_score else '#e74c3c' for s in scores]

    fig, axes = plt.subplots(1, 3, figsize=(18, max(4, M * 0.5 + 2)))
    fig.patch.set_facecolor('#1a1a2e')
    for ax in axes:
        ax.set_facecolor('#16213e')
        ax.tick_params(colors='white')
        for spine in ax.spines.values():
            spine.set_color('#0f3460')

    short_claims = [f"C{i}: {c[:35]}..." if len(c) > 35 else f"C{i}: {c}" for i, c in enumerate(claims)]
    y_pos = np.arange(M)

    # Panel 1: SeSE scores
    bars = axes[0].barh(y_pos, scores, color=colors, height=0.65, alpha=0.85)
    axes[0].axvline(median_score, color='#f39c12', linestyle='--', lw=1.5, label='Median')
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(short_claims, color='white', fontsize=8)
    axes[0].set_xlabel('SeSE Score', color='white')
    axes[0].set_title('Claim Uncertainty (SeSE)\n↑ higher = more uncertain', color='white', fontweight='bold')
    axes[0].legend(facecolor='#16213e', labelcolor='white', fontsize=8)
    for bar, score in zip(bars, scores):
        axes[0].text(score + 0.001, bar.get_y() + bar.get_height()/2,
                     f'{score:.3f}', va='center', color='white', fontsize=7)

    # Panel 2: Support ratio
    s_colors = [plt.cm.RdYlGn(r) for r in support_ratio]
    axes[1].barh(y_pos, support_ratio, color=s_colors, height=0.65, alpha=0.85)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(short_claims, color='white', fontsize=8)
    axes[1].set_xlabel('Support Ratio (responses entailing claim)', color='white')
    axes[1].set_title('Response Support Ratio\n↑ higher = more supported', color='white', fontweight='bold')
    axes[1].set_xlim(0, 1)
    for i, (sr, sp) in enumerate(zip(support_ratio, support)):
        axes[1].text(sr + 0.01, i, f'{sp}/{len(sampled_responses)}', va='center', color='white', fontsize=7)

    # Panel 3: Scatter SeSE vs Support
    scatter = axes[2].scatter(support_ratio, scores, c=scores, cmap='RdYlGn_r',
                               s=120, zorder=3, edgecolors='white', lw=0.5)
    for i, (sr, sc_val) in enumerate(zip(support_ratio, scores)):
        axes[2].annotate(f'C{i}', (sr, sc_val), textcoords='offset points',
                         xytext=(5, 3), fontsize=8, color='white')
    axes[2].set_xlabel('Support Ratio', color='white')
    axes[2].set_ylabel('SeSE Score', color='white')
    axes[2].set_title('SeSE vs Support Ratio\n(bottom-right = low UQ, high support)', color='white', fontweight='bold')
    axes[2].axhline(median_score, color='#f39c12', linestyle='--', lw=1, alpha=0.7)
    axes[2].axvline(0.5, color='#3498db', linestyle='--', lw=1, alpha=0.7)
    cbar = plt.colorbar(scatter, ax=axes[2])
    cbar.set_label('SeSE Score', color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

    plt.suptitle('SeSE Long-Form Uncertainty Quantification Results',
                 color='white', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


plot_sese_results(claims, sese_scores, adj_matrix, sampled_responses)

In [ ]:
def visualize_bipartite_graph(G_cr, r_nodes, c_nodes, claims, sese_scores):
    """Visualize the claim-response bipartite graph with SeSE-colored claim nodes."""
    median = np.nanmedian(list(sese_scores.values()))

    pos = {}
    for i, rn in enumerate(r_nodes):
        pos[rn] = (0, i - len(r_nodes) / 2)
    for j, cn in enumerate(c_nodes):
        pos[cn] = (3, j - len(c_nodes) / 2)

    r_color = '#3498db'
    c_colors = [
        '#2ecc71' if sese_scores.get(claims[int(cn.split('_')[1])], float('inf')) < median
        else '#e74c3c'
        for cn in c_nodes
    ]
    node_colors = [r_color] * len(r_nodes) + c_colors
    node_sizes = [300] * len(r_nodes) + [400] * len(c_nodes)

    fig, ax = plt.subplots(figsize=(12, max(8, max(len(r_nodes), len(c_nodes)) * 0.8)))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    nx.draw_networkx(
        G_cr,
        pos=pos,
        node_color=node_colors,
        node_size=node_sizes,
        font_size=7,
        font_color='white',
        edge_color='#7f8c8d',
        alpha=0.85,
        ax=ax,
        arrows=False,
    )

    # Add claim text annotations
    for j, (cn, claim) in enumerate(zip(c_nodes, claims)):
        score = sese_scores.get(claim, float('nan'))
        short = claim[:40] + '...' if len(claim) > 40 else claim
        ax.annotate(f"{short}\n(SeSE={score:.3f})",
                    xy=(3, j - len(c_nodes) / 2),
                    xytext=(3.3, j - len(c_nodes) / 2),
                    fontsize=6, color='white', va='center')

    ax.set_title("Claim-Response Bipartite Graph\nBlue=Responses  Green=Low-UQ Claims  Red=High-UQ Claims",
                 color='white', fontsize=11, fontweight='bold')
    r_patch = mpatches.Patch(color='#3498db', label='Response nodes')
    g_patch = mpatches.Patch(color='#2ecc71', label='Low uncertainty claim')
    e_patch = mpatches.Patch(color='#e74c3c', label='High uncertainty claim')
    ax.legend(handles=[r_patch, g_patch, e_patch], facecolor='#16213e',
              labelcolor='white', fontsize=8, loc='lower left')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


visualize_bipartite_graph(G_cr, r_nodes, c_nodes, claims, sese_scores)

## 9. Full Pipeline Wrapper

In [ ]:
def sese_longform_pipeline(
    question: str,
    cfg: SeSEConfig = cfg,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    End-to-end SeSE pipeline for long-form uncertainty quantification.

    Args:
        question: The input query/prompt for the LLM.
        cfg: SeSEConfig instance.
        verbose: Whether to print progress.

    Returns:
        DataFrame with columns: claim, sese_score, support, support_ratio, uncertain
    """
    if verbose: print(f"\n{'='*60}\nSeSE Pipeline\nQuestion: {question}\n{'='*60}")

    # Step 1: Sample responses
    if verbose: print("Step 1: Sampling responses...")
    greedy, samples = sample_responses(
        question,
        n_samples=cfg.n_samples,
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        top_k=cfg.top_k,
        max_new_tokens=cfg.max_new_tokens,
    )
    if verbose: print(f"  Greedy response: {greedy[:100]}...")

    # Step 2: Decompose claims
    if verbose: print("Step 2: Decomposing claims...")
    claims_ = decompose_claims(greedy)
    if verbose: print(f"  {len(claims_)} claims extracted.")

    # Step 3: Build bipartite graph
    if verbose: print("Step 3: Building bipartite graph...")
    G_cr_, adj_, r_ns, c_ns = build_bipartite_graph(samples, claims_)

    # Step 4 & 5: Compute SeSE
    if verbose: print("Steps 4-5: Computing SeSE scores...")
    scores, _ = compute_sese_longform(G_cr_, c_ns, claims_, K=cfg.K)

    # Build results DataFrame
    median_score = np.nanmedian(list(scores.values()))
    rows = []
    for j, claim in enumerate(claims_):
        sc = scores.get(claim, float('nan'))
        sup = int(adj_[:, j].sum())
        rows.append({
            "claim": claim,
            "sese_score": sc,
            "support": sup,
            "support_ratio": sup / cfg.n_samples,
            "uncertain": sc >= median_score,
        })

    df = pd.DataFrame(rows).sort_values("sese_score")

    if verbose:
        print("\n=== Results ===")
        display_df = df.copy()
        display_df["claim"] = display_df["claim"].apply(lambda x: x[:70] + '...' if len(x) > 70 else x)
        display_df["sese_score"] = display_df["sese_score"].round(4)
        display_df["support_ratio"] = display_df["support_ratio"].round(2)
        print(display_df.to_string(index=False))

    return df


# ─── Run on your own question ─────────────────────────────────────────────────
results_df = sese_longform_pipeline("Tell me about Albert Einstein's scientific contributions")
print("\nDone! Results stored in `results_df`.")

## 10. Final Summary

In [ ]:
def print_hallucination_report(df: pd.DataFrame):
    """Print a clean hallucination risk report."""
    print("\n" + "▓"*65)
    print("   SESE HALLUCINATION RISK REPORT")
    print("▓"*65)

    certain = df[~df["uncertain"]].sort_values("sese_score")
    uncertain = df[df["uncertain"]].sort_values("sese_score", ascending=False)

    print(f"\n✅ LOW UNCERTAINTY CLAIMS ({len(certain)}/{len(df)}) — likely factual:")
    for _, row in certain.iterrows():
        print(f"   [{row['sese_score']:.4f}] {row['claim']}")

    print(f"\n⚠️  HIGH UNCERTAINTY CLAIMS ({len(uncertain)}/{len(df)}) — potential hallucinations:")
    for _, row in uncertain.iterrows():
        print(f"   [{row['sese_score']:.4f}] {row['claim']}")

    overall = df['sese_score'].mean()
    print(f"\n📊 Overall mean SeSE: {overall:.4f}")
    print(f"   Uncertainty rate: {df['uncertain'].mean()*100:.1f}%")
    print("▓"*65)


print_hallucination_report(results_df)

# Optional: export results
results_df.to_csv("sese_results.csv", index=False)
print("\nResults saved to sese_results.csv")